In [ ]:
!pip install 
!pip install folium

In [ ]:
#imports
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer
import folium
import os

In [ ]:
BASE_URL = "https://sinca.mma.gob.cl/cgi-bin/APUB-MMA/apub.tsindico2.cgi"

# Estaciones del gran concepción que miden MP2,5  -> despues podria medir para toda la region
# codigo: (nombre, comuna)
ESTACIONES = {
    "854": ("Punteras", "Chiguayante"),
    "827": ("Kingston College", "Concepcion"),
    "831": ("Cerro Merquin", "Coronel"),
    "878": ("Coronel Norte", "Coronel"),
    "880": ("Escuadron ENEL", "Coronel"),
    "881": ("Escuadron ENESA", "Coronel"),
    "879": ("Lagunillas ENEL", "Coronel"),
    "859": ("Lota rural", "Coronel"),
    "860": ("Lota urbana", "Coronel"),
    "805": ("Bocatoma", "Hualpen"),
    "838": ("ENAP Price", "Hualpen"),
    "804": ("JUNJI", "Hualpen"),
    "841": ("Hualqui", "Hualqui"),
    "802": ("Consultorio San Vicente", "Talcahuano"),
    "807": ("Indura", "Talcahuano"),
    "806": ("Inpesca", "Talcahuano"),
}

# Contaminantes que queremos traer, con su código SINCA
# nombre_legible: codigo_sinca
CONTAMINANTES = {
    "mp2.5": "PM25",
    "mp10":  "PM10",
    "so2":   "0001",
    "no":    "0002",
    "no2":   "0003",
    "co":    "0004",
    "o3":    "0008",
    "nox":   "0NOX",
}

In [ ]:
def pedir_sinca(region,estacion_cod, parametro, desde, hasta):
    """
    Pide datos a SINCA y devuelve en texto crudo.
        - region: codigo de region (ej: bio bio -> RVIII)
        - estadion_cod : Código de la estacion (ej: "827" para Kingston College)
        - parametro: que medir (ej: "PM25", para MP2,5)
        - desde, hasta: fechas en formato AAMMDD (ej: "240101" => 1 enero 2024)

        sinca response => 240101;0100;23;;;
            - 240101 => 1 de enero 2024
            - 0100   => 01:00 hora 
            - 23;;     => 23 µg/m³ de MP2,5
             │ │ │
             │ │ └─ posición 3: NO validado
             │ └─── posición 2: preliminar
             └───── posición 1: validado
    """
    params = {
        "outtype": "xcl", # fromato de salida (texto/CSV)
        "macro": f"./{region}/{estacion_cod}/Cal/{parametro}/{parametro}.horario.horario.ic",
        "from": desde,
        "to": hasta,
        "path": "/usr/airviro/data/CONAMA/",
        "lang": "esp"
    }
    respuesta = requests.get(BASE_URL, params = params, timeout = 60)
    return respuesta.text

##Test de la funcion    
##texto = pedir_sinca("RVIII","827", "PM25", "240101", "240107")
##print(texto[:500])

In [ ]:
def parsear_sinca(texto):
    """
    Convierte el texto crudo de SINCA en un tabla (dataframe).
    columnas de salida: timestamp, valor, estado
    """
    estados = ["validado","preliminar","no_validado"]
    filas = []
    
    for linea in texto.splitlines()[1:]:
        if not linea.strip():
            continue # saltamos las lineas vacias

        campos = linea.split(";")
        
        if len(campos) < 3:
            continue # saltamos linea incompleta

        fecha = campos[0].strip() # ej: "240101"
        hora = campos[1].strip()  # ej: "0100"
        timestamp = pd.to_datetime(fecha + hora[:2], format="%y%m%d%H")
        
        #buscamos el valor en las 3 posiciones posibles 2,3,4
        valor = None
        estado = None

        for i, nombre_estado in enumerate(estados):
            pos = 2 + i
            if pos < len(campos) and campos[pos].strip():
                valor = float(campos[pos].replace(",", "."))  # valor = float(campos[pos])
                estado = nombre_estado
                break

        if valor is None:
            continue #la hora no tenia medicion
    
        filas.append({"timestamp": timestamp, "valor": valor, "estado": estado})
    
    df = pd.DataFrame(filas)
    return df
## Test
##tabla = parsear_sinca(texto)
##print(tabla)

In [ ]:
def traer_estaciones(estaciones, contaminantes, desde, hasta, region="RVIII"):
    """
    Recorre un diccionario de estaciones, pide cada una a SINCA,
    le agrega nombre/comuna, y junta todo en una sola tabla.
    """
    tablas = []

    for codigo, (nombre, comuna) in estaciones.items():
        for nombre_cont, codigo_cont in contaminantes.items():   # NUEVO: bucle de contaminantes

            print(f"Trayendo {nombre} / {nombre_cont}...", end=" ")
        
            # pedir y parsear (reusamos las funciones que ya hicimos)
            texto = pedir_sinca(region, codigo, codigo_cont, desde, hasta)
            tabla = parsear_sinca(texto)

            if tabla is None or len(tabla) == 0:
                print("sin datos")
                continue

            # agregar de qué estación es cada fila
            tabla["codigo"] = codigo
            tabla["estacion"] = nombre
            tabla["comuna"] = comuna
            tabla["parametro"] = nombre_cont

            tablas.append(tabla)
            print(f"{len(tabla)} registros")

    # juntar todas las tablas en una sola
    if not tablas:
        return None
    return pd.concat(tablas, ignore_index=True)


In [ ]:
os.makedirs("data", exist_ok=True)
ARCHIVO = "data/contaminantes_gran_concepcion.csv"   # nombre nuevo (ya no es solo mp25)

if os.path.exists(ARCHIVO):
    datos = pd.read_csv(ARCHIVO, parse_dates=["timestamp"])
    print(f"Cargado desde {ARCHIVO}: {len(datos)} filas")
else:
    print("Descargando todo el histórico de todos los contaminantes... (tarda varios minutos)")
    datos = traer_estaciones(ESTACIONES, CONTAMINANTES, "110101", "260711")  # ahora CONTAMINANTES
    datos.to_csv(ARCHIVO, index=False)
    print(f"\nGuardado en {ARCHIVO}: {len(datos)} filas")

print(f"Estaciones: {datos['estacion'].nunique()}")
print(f"Parámetros: {datos['parametro'].unique()}")
print(f"Rango: {datos['timestamp'].min()} a {datos['timestamp'].max()}")

In [ ]:

# Carpeta donde guardar
os.makedirs("data", exist_ok=True)
ARCHIVO = "data/mp25_gran_concepcion.csv"

# Si el archivo ya existe, lo cargamos (rápido). Si no, lo descargamos (lento).
if os.path.exists(ARCHIVO):
    datos = pd.read_csv(ARCHIVO, parse_dates=["timestamp"])
    print(f"Cargado desde {ARCHIVO}: {len(datos)} filas")
else:
    print("Descargando todo el histórico... (esto tarda unos minutos)")
    datos = traer_estaciones(ESTACIONES, "PM25", "110101", "260711")  # 2011 a hoy
    datos.to_csv(ARCHIVO, index=False)
    print(f"\nGuardado en {ARCHIVO}: {len(datos)} filas")

print(f"Estaciones: {datos['estacion'].nunique()}")
print(f"Rango: {datos['timestamp'].min()} a {datos['timestamp'].max()}")

In [ ]:
# Convertir a parquet (más liviano y rápido)
# Cargar el CSV (tarda, son 500 MB)
print("Cargando CSV...")
datos = pd.read_csv("data/contaminantes_gran_concepcion.csv", parse_dates=["timestamp"])
print(f"Cargado: {len(datos):,} filas")

# Guardar como parquet
print("Guardando parquet...")
datos.to_parquet("data/contaminantes_gran_concepcion.parquet", index=False)

# Comparar tamaños
csv_mb = os.path.getsize("data/contaminantes_gran_concepcion.csv") / 1024**2
pq_mb  = os.path.getsize("data/contaminantes_gran_concepcion.parquet") / 1024**2
print(f"\nCSV:     {csv_mb:.0f} MB")
print(f"Parquet: {pq_mb:.0f} MB")
print(f"Reducción: {100*(1-pq_mb/csv_mb):.0f}%")

# Testeo de la data 

In [ ]:
# Creamos la figura
fig, ax = plt.subplots(figsize=(14, 6))

# Dibujamos una línea por cada estación
for nombre in datos["estacion"].unique():
    # filtramos solo las filas de esa estación
    d = datos[datos["estacion"] == nombre]
    ax.plot(d["timestamp"], d["valor"], label=nombre, linewidth=1)

# Etiquetas y leyenda
ax.set_title("MP2,5 por estación — primera semana de enero 2024")
ax.set_xlabel("Fecha y hora")
ax.set_ylabel("MP2,5 (µg/m³)")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Coordenadas UTM (huso 18 Sur). 
# "V" = verificada desde ficha oficial SINCA. "A" = aproximada (afinar después).
UTM_ESTACIONES = {
    "854": (674923, 5911833),   # Punteras, Chiguayante        - V
    "827": (673817, 5927247),   # Kingston College, Concepcion - V
    "831": (663500, 5905000),   # Cerro Merquin, Coronel       - A
    "878": (664000, 5915000),   # Coronel Norte, Coronel       - A
    "880": (662500, 5903000),   # Escuadron ENEL, Coronel      - A
    "881": (662400, 5902800),   # Escuadron ENESA, Coronel     - A
    "879": (663000, 5904000),   # Lagunillas ENEL, Coronel     - A
    "859": (664500, 5895000),   # Lota rural, Coronel          - A
    "860": (664200, 5894500),   # Lota urbana, Coronel         - A
    "805": (669000, 5931000),   # Bocatoma, Hualpen            - A
    "838": (668500, 5930500),   # ENAP Price, Hualpen          - A
    "804": (669500, 5931500),   # JUNJI, Hualpen               - A
    "841": (682000, 5910000),   # Hualqui, Hualqui             - A
    "802": (668000, 5933000),   # Consultorio San Vicente, Talcahuano - A
    "807": (670000, 5932000),   # Indura, Talcahuano           - A
    "806": (669000, 5932500),   # Inpesca, Talcahuano          - A
}

# Convertir todas a lat/lon
COORDENADAS = {}
for cod, (e, n) in UTM_ESTACIONES.items():
    COORDENADAS[cod] = utm_a_latlon(e, n)

for cod, (lat, lon) in COORDENADAS.items():
    print(f"{cod}: {lat}, {lon}")

In [ ]:
# Calcular el MP2,5 promedio de cada estación (de los datos que ya tienes)
promedios = datos.groupby(["codigo", "estacion"])["valor"].mean().reset_index()

# Centro del mapa: Gran Concepción
mapa = folium.Map(location=[-36.90, -73.10], zoom_start=11)

# Agregar un punto por cada estación
for _, fila in promedios.iterrows():
    cod = fila["codigo"]
    if cod not in COORDENADAS:
        continue
    lat, lon = COORDENADAS[cod]
    valor = fila["valor"]

    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        popup=f"{fila['estacion']}: {valor:.1f} µg/m³",
        color="black",
        weight=1,
        fill=True,
        fill_color="red",
        fill_opacity=0.6,
    ).add_to(mapa)

# Mostrar el mapa
mapa

In [ ]:
#from src.config import ESTACIONES, CONTAMINANTES
#print(len(ESTACIONES), len(CONTAMINANTES))

from src.config import ESTACIONES, CONTAMINANTES
from src.sinca import traer_estaciones

datos = traer_estaciones(ESTACIONES, CONTAMINANTES, "110101", "260711")

In [ ]:
import os
os.makedirs("../data", exist_ok=True)
datos.to_parquet("../data/crudo.parquet", index=False)
print(f"Guardado: {len(datos):,} filas")

In [ ]:
from src.meteo import utm_a_latlon, pedir_openmeteo, traer_meteo, agregar_derivadas
from src.config import UTM_ESTACIONES
coords = utm_a_latlon(UTM_ESTACIONES)
print(coords["878"])   # Coronel Norte → debería dar (~-36.9, ~-73.15)

In [ ]:
import sys; sys.path.append("..")
from src.config import ESTACIONES, METEO_MODELO, METEO_CELDA
print(ESTACIONES["859"])
print(METEO_MODELO, METEO_CELDA)

In [ ]:
import sys; sys.path.append("..")
from src.meteo import pedir_openmeteo

t = pedir_openmeteo(-37.009, -73.1491, "2025-07-01", "2025-07-02")   # Coronel Norte
print(len(t), "horas")
print(t[["timestamp", "temperature_2m", "boundary_layer_height", "celda_lat", "celda_lon", "celda_elev"]].head(3))

In [ ]:
import sys; sys.path.append("..")
from src.meteo import pedir_por_tramos

t = pedir_por_tramos(-37.009, -73.1491, "2024-12-30", "2025-01-02")   # cruza dos años
print(len(t), "horas")
print(t["timestamp"].min(), "→", t["timestamp"].max())
print(t[["celda_lat", "celda_lon"]].drop_duplicates())

In [ ]:
import sys; sys.path.append("..")
from src.config import ESTACIONES, UTM_ESTACIONES
from src.meteo import utm_a_latlon, traer_meteo

coords = utm_a_latlon(UTM_ESTACIONES)
dos = {c: ESTACIONES[c] for c in ["878", "841"]}   # Coronel Norte y Hualqui
t = traer_meteo(dos, coords, "2025-07-01", "2025-07-02")
print(t.groupby("estacion")["temperature_2m"].mean().round(2))

In [ ]:
import src.meteo as m
print([f for f in dir(m) if not f.startswith("__")])

In [ ]:
import sys; sys.path.append("..")
from src.config import ESTACIONES, UTM_ESTACIONES
from src.meteo import utm_a_latlon, traer_meteo, informe_resolucion

coords = utm_a_latlon(UTM_ESTACIONES)
dos = {c: ESTACIONES[c] for c in ["878", "841"]}
t = traer_meteo(dos, coords, "2025-07-01", "2025-07-02")

inf = informe_resolucion(t)
print(inf["estaciones"][["estacion", "dist_celda_km", "celda_elev", "id_celda"]])
print(inf["variables"])